# Enterprise Governance - Analytics Notebook

This notebook provides interactive analysis of AI coding usage data for enterprise governance.

## Setup

In [ ]:
# Install dependencies if needed
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn

In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries loaded successfully!')

## 1. Load Data

Load AI coding task data from the mock-data directory or real data.

In [ ]:
# Configuration
USE_MOCK_DATA = True  # Set to False to use real data

if USE_MOCK_DATA:
    DATA_DIR = Path('../mock-data')
else:
    DATA_DIR = Path.home() / '.claude' / 'todos'

print(f'Loading data from: {DATA_DIR}')

In [ ]:
def load_task_data(data_dir: Path) -> pd.DataFrame:
    """Load all task JSON files into a DataFrame."""
    tasks = []
    
    for file_path in data_dir.glob('*.json'):
        try:
            with open(file_path, 'r') as f:
                content = json.load(f)
            
            entries = content if isinstance(content, list) else [content]
            for entry in entries:
                entry['_source_file'] = file_path.name
                tasks.append(entry)
        except Exception as e:
            print(f'Warning: Could not read {file_path}: {e}')
    
    return pd.DataFrame(tasks)

# Load data
df = load_task_data(DATA_DIR)
print(f'Loaded {len(df)} tasks from {df["_source_file"].nunique()} sessions')

In [ ]:
# Preprocess data
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and enrich the DataFrame."""
    df = df.copy()
    
    # Parse timestamps
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        df['date'] = df['timestamp'].dt.date
        df['hour'] = df['timestamp'].dt.hour
        df['day_of_week'] = df['timestamp'].dt.day_name()
    
    # Extract tokens
    if 'tokens' in df.columns:
        df['tokens_input'] = df['tokens'].apply(
            lambda x: x.get('input', 0) if isinstance(x, dict) else 0
        )
        df['tokens_output'] = df['tokens'].apply(
            lambda x: x.get('output', 0) if isinstance(x, dict) else 0
        )
        df['tokens_total'] = df['tokens_input'] + df['tokens_output']
    else:
        df['tokens_input'] = 0
        df['tokens_output'] = 0
        df['tokens_total'] = 0
    
    # Calculate costs (LLM API pricing)
    cost_input = {
        'claude-haiku-3-5-20241022': 0.25,
        'claude-sonnet-4-20250514': 3.0,
        'claude-opus-4-20250514': 15.0
    }
    cost_output = {
        'claude-haiku-3-5-20241022': 1.25,
        'claude-sonnet-4-20250514': 15.0,
        'claude-opus-4-20250514': 75.0
    }
    
    df['cost_input'] = df.apply(
        lambda r: (r['tokens_input'] / 1_000_000) * cost_input.get(r.get('model', ''), 3.0),
        axis=1
    )
    df['cost_output'] = df.apply(
        lambda r: (r['tokens_output'] / 1_000_000) * cost_output.get(r.get('model', ''), 15.0),
        axis=1
    )
    df['cost_total'] = df['cost_input'] + df['cost_output']
    
    # Fill missing values
    df['status'] = df.get('status', pd.Series(['unknown'] * len(df))).fillna('unknown')
    df['type'] = df.get('type', pd.Series(['general'] * len(df))).fillna('general')
    df['model'] = df.get('model', pd.Series(['unknown'] * len(df))).fillna('unknown')
    
    return df

df = preprocess_data(df)
df.head()

## 2. Executive Summary

In [ ]:
# Key metrics
summary = {
    'Total Tasks': len(df),
    'Total Sessions': df['_source_file'].nunique(),
    'Total Input Tokens': f"{df['tokens_input'].sum():,}",
    'Total Output Tokens': f"{df['tokens_output'].sum():,}",
    'Total Cost': f"${df['cost_total'].sum():.2f}",
    'Avg Cost/Task': f"${df['cost_total'].mean():.4f}",
    'Completion Rate': f"{(df['status'] == 'completed').mean() * 100:.1f}%"
}

print('=' * 60)
print('           EXECUTIVE SUMMARY')
print('=' * 60)
for key, value in summary.items():
    print(f'{key:25} {value}')
print('=' * 60)

## 3. Status Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
status_counts = df['status'].value_counts()
colors = ['#10B981', '#F59E0B', '#EF4444', '#6B7280']  # green, yellow, red, gray
axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors[:len(status_counts)], startangle=90)
axes[0].set_title('Task Status Distribution', fontsize=14, fontweight='bold')

# Bar chart
status_counts.plot(kind='bar', ax=axes[1], color=colors[:len(status_counts)])
axes[1].set_title('Task Count by Status', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Model Usage Analysis

In [ ]:
# Model usage breakdown
model_stats = df.groupby('model').agg({
    'content': 'count',
    'tokens_input': 'sum',
    'tokens_output': 'sum',
    'cost_total': 'sum'
}).rename(columns={'content': 'task_count'})

model_stats['avg_cost'] = model_stats['cost_total'] / model_stats['task_count']
model_stats['pct_of_tasks'] = model_stats['task_count'] / model_stats['task_count'].sum() * 100

print('Model Usage Statistics:')
print(model_stats.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Tasks by model
model_stats['task_count'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Tasks by Model', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Task Count')
axes[0].tick_params(axis='x', rotation=45)

# Cost by model
model_stats['cost_total'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Cost by Model ($)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Total Cost ($)')
axes[1].tick_params(axis='x', rotation=45)

# Avg cost per task by model
model_stats['avg_cost'].plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Avg Cost per Task by Model ($)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Avg Cost ($)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Task Type Analysis

In [ ]:
# Task type distribution
type_stats = df.groupby('type').agg({
    'content': 'count',
    'cost_total': 'sum',
    'tokens_total': 'sum'
}).rename(columns={'content': 'task_count'}).sort_values('task_count', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
type_stats['task_count'].plot(kind='barh', ax=ax, color='mediumpurple')
ax.set_title('Tasks by Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Task Count')
ax.set_ylabel('Task Type')
plt.tight_layout()
plt.show()

## 6. Time-Based Analysis

In [ ]:
if 'timestamp' in df.columns and df['timestamp'].notna().any():
    # Daily trend
    daily = df.groupby('date').agg({
        'content': 'count',
        'cost_total': 'sum',
        'tokens_total': 'sum'
    }).rename(columns={'content': 'task_count'})
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Tasks over time
    daily['task_count'].plot(ax=axes[0], marker='o', linewidth=2, color='steelblue')
    axes[0].set_title('Daily Task Volume', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Tasks')
    axes[0].grid(True, alpha=0.3)
    
    # Cost over time
    daily['cost_total'].plot(ax=axes[1], marker='s', linewidth=2, color='coral')
    axes[1].set_title('Daily Cost ($)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Cost ($)')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('No timestamp data available for time-based analysis')

In [ ]:
# Hourly pattern (if timestamps available)
if 'hour' in df.columns and df['hour'].notna().any():
    hourly = df.groupby('hour').agg({'content': 'count'}).rename(columns={'content': 'task_count'})
    
    fig, ax = plt.subplots(figsize=(14, 5))
    hourly['task_count'].plot(kind='bar', ax=ax, color='teal')
    ax.set_title('Task Distribution by Hour of Day', fontsize=14, fontweight='bold')
    ax.set_xlabel('Hour')
    ax.set_ylabel('Task Count')
    plt.tight_layout()
    plt.show()

## 7. Cost Analysis

In [ ]:
# Cost breakdown
cost_breakdown = {
    'Total Cost': df['cost_total'].sum(),
    'Input Cost': df['cost_input'].sum(),
    'Output Cost': df['cost_output'].sum(),
    'Avg Cost/Task': df['cost_total'].mean(),
    'Max Cost (single task)': df['cost_total'].max(),
    'Median Cost': df['cost_total'].median()
}

print('Cost Breakdown:')
for key, value in cost_breakdown.items():
    print(f'{key:25} ${value:.4f}')

In [ ]:
# Cost distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall cost distribution
df['cost_total'].hist(bins=30, ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Cost Distribution (All Tasks)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Cost ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['cost_total'].mean(), color='red', linestyle='--', label=f'Mean: ${df["cost_total"].mean():.4f}')
axes[0].legend()

# Input vs Output cost
cost_comparison = pd.DataFrame({
    'Input': [df['cost_input'].sum()],
    'Output': [df['cost_output'].sum()]
}).T
cost_comparison.plot(kind='bar', ax=axes[1], legend=False, color=['steelblue', 'coral'])
axes[1].set_title('Input vs Output Cost', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cost ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 8. Pattern Detection

In [ ]:
# Extract task patterns from content
if 'content' in df.columns:
    patterns = df['content'].dropna().apply(
        lambda x: ' '.join(str(x).split()[:3])
    )
    
    pattern_counts = patterns.value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    pattern_counts.plot(kind='barh', ax=ax, color='mediumseagreen')
    ax.set_title('Top 10 Task Patterns (First 3 Words)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.set_ylabel('Pattern')
    plt.tight_layout()
    plt.show()

## 9. Health Score Calculation

In [ ]:
# Calculate health metrics
total_tasks = len(df)
completed = (df['status'] == 'completed').sum()
failed = (df['status'] == 'failed').sum()
in_progress = (df['status'] == 'in_progress').sum()

# Health score components
completion_rate = completed / total_tasks * 100 if total_tasks > 0 else 0
failure_rate = failed / total_tasks * 100 if total_tasks > 0 else 0

# Calculate overall health score (0-100)
health_score = min(100, max(0, completion_rate - (failure_rate * 2)))

# Determine health status
if health_score >= 80:
    health_status = 'HEALTHY'
    health_color = 'green'
elif health_score >= 60:
    health_status = 'WARNING'
    health_color = 'orange'
else:
    health_status = 'CRITICAL'
    health_color = 'red'

print('=' * 50)
print(f'SYSTEM HEALTH: {health_status}')
print('=' * 50)
print(f'Health Score:     {health_score:.1f}/100')
print(f'Completion Rate:  {completion_rate:.1f}%')
print(f'Failure Rate:     {failure_rate:.1f}%')
print(f'In Progress:      {in_progress} tasks')
print('=' * 50)

## 10. Forecasting (Simple Linear Trend)

In [ ]:
if 'date' in df.columns and df['date'].notna().any():
    from sklearn.linear_model import LinearRegression
    
    # Prepare daily data
    daily_costs = df.groupby('date')['cost_total'].sum().reset_index()
    daily_costs['day_num'] = range(len(daily_costs))
    
    if len(daily_costs) >= 3:
        # Fit linear model
        X = daily_costs[['day_num']].values
        y = daily_costs['cost_total'].values
        
        model = LinearRegression()
        model.fit(X, y)
        
        # Forecast next 7 days
        future_days = np.array([[daily_costs['day_num'].max() + i] for i in range(1, 8)])
        forecast = model.predict(future_days)
        
        # Plot
        fig, ax = plt.subplots(figsize=(14, 6))
        
        ax.plot(daily_costs['day_num'], y, 'o-', label='Historical', color='steelblue', linewidth=2)
        ax.plot(future_days, forecast, 's--', label='Forecast', color='coral', linewidth=2)
        ax.fill_between(future_days.flatten(), forecast * 0.8, forecast * 1.2, alpha=0.2, color='coral')
        
        ax.set_title('Cost Forecast (7-Day)', fontsize=14, fontweight='bold')
        ax.set_xlabel('Day')
        ax.set_ylabel('Cost ($)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f'\n7-Day Cost Forecast: ${forecast.sum():.2f}')
        print(f'Daily Trend: ${model.coef_[0]:.4f}/day')
        print(f'30-Day Projection: ${model.coef_[0] * 30 + model.intercept_ * 30:.2f}')
    else:
        print('Not enough data points for forecasting (need at least 3 days)')
else:
    print('No timestamp data available for forecasting')

## 11. Export Data

In [ ]:
# Export summary to CSV
# df.to_csv('../output/task_data_export.csv', index=False)
# print('Data exported to output/task_data_export.csv')

# Summary statistics
summary_df = pd.DataFrame({
    'Metric': list(summary.keys()),
    'Value': list(summary.values())
})
print('\nSummary for export:')
print(summary_df.to_string(index=False))

---

## Appendix: Model Pricing Reference

| Model | Input ($/M tokens) | Output ($/M tokens) |
|-------|-------------------|--------------------|
| Haiku 3.5 | $0.25 | $1.25 |
| Sonnet 4 | $3.00 | $15.00 |
| Opus 4 | $15.00 | $75.00 |

*Pricing as of January 2025. Check your provider's website for current rates.*